# 08 · Spark and Beam

*The same station job in two more frameworks, and what each one is built for.*

Notebook 07 ran the pipeline on Ray. This notebook takes the two steps that matter and writes them in **Apache Spark** and **Apache Beam**. The point isn't a benchmark on our small data. Each framework was designed around a different idea of what a data job is, and those ideas decide where each one is easy and where it's awkward. Writing the same job in each is the quickest way to feel that.

- **Spark** thinks in tables. You describe a query over DataFrames (select, join, group) and its optimizer turns it into a plan of stages split at **shuffles**, the points where rows must move between machines because they're grouped by a new key. It grew out of large batch analytics and SQL.
- **Beam** thinks in collections flowing through a graph. A **PCollection** is a dataset that may be finite (batch) or never-ending (stream), and every element carries an **event time**. The same pipeline code runs on different engines called **runners**: Google Cloud Dataflow, Flink, Spark or a local DirectRunner. It grew out of Google's streaming work and its big idea is that batch is just a stream that ends.

Two steps from our pipeline show the difference best. One is the **neighbour difference**, which compares each station with others at the same hour and so needs a join across stations (a shuffle). The other is **late data**: a station-hour that arrives after its day has already been checked.

**Read first:** Spark's [SQL and DataFrames guide](https://spark.apache.org/docs/latest/sql-programming-guide.html) (overview and "Getting started"), Beam's [Programming guide](https://beam.apache.org/documentation/programming-guide/) sections 1–3 and 7 (windowing), and the classic [Streaming 101](https://www.oreilly.com/radar/the-world-beyond-batch-streaming-101/) essay. About 60 minutes.

Spark needs a Java 17 or 21 runtime (`brew install openjdk@21` on macOS).

In [1]:
# Setup: identical in every notebook. Data lives in ../data, which git ignores.
import os
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
os.makedirs(DATA_DIR, exist_ok=True)
print("Data directory:", DATA_DIR)

Data directory: /Users/ashu/Workspace/weather-data-lab/data


In [2]:
import warnings; warnings.filterwarnings("ignore")
import time
import sys, logging
os.environ["GRPC_VERBOSITY"] = "ERROR"                          # quiet Beam's internal RPC logs
logging.getLogger().setLevel(logging.ERROR)
os.environ["PYSPARK_PYTHON"] = sys.executable                 # workers use this notebook's Python
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home")   # Homebrew's Java 21
import pandas as pd
from pathlib import Path
from wxlab import qa

STD = Path(DATA_DIR) / "standardized" / "isd"
points = pd.read_parquet(Path(DATA_DIR) / "reference" / "station_points.parquet")

## Part 1 · Spark

### A session and a table

A `SparkSession` is the entry point. Locally it runs the driver and the workers as threads in one Java process. Reading the standardized Parquet tree gives a DataFrame. We use one year, 2023, which keeps the whole run under about 4 GB of memory on a laptop. Spark reads only the files and columns the query needs, and the `year=` folders become a column it can prune on (**partition pruning**).

In [3]:
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder.master("local[4]").appName("wxlab")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "16")      # the default of 200 is sized for clusters
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.driver.memory", "3g")               # Java heap cap; one year peaks near 4 GB in total
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
obs = spark.read.parquet(str(STD)).where(F.col("year") == 2023)      # one year keeps a laptop comfortable
obs = obs.select("station_id", "time", "air_temperature", "year").where(F.col("air_temperature").isNotNull())
print(obs.count(), "station-hours")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/09 11:44:00 WARN Utils: Your hostname, Ashus-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 172.20.20.20 instead (on interface en0)
26/11/09 11:44:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/09 11:44:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


1286813 station-hours


### The neighbour difference as a join

In pandas notebook 05 computed, for each station-hour, the station's temperature minus the median of its neighbours at the same hour. In Spark that's a join: station pairs from a small table, joined to the observations twice, on station and on time. Spark broadcasts the small pairs table to every worker, but the two sides of the time join must be **shuffled** so equal hours land together.

In [4]:
nbrs = qa.neighbours(points)                         # station → its neighbours, as in notebook 05
pairs = spark.createDataFrame([(s, t) for s, ns in nbrs.items() for t in ns], ["station_id", "nbr_id"])
o = obs.alias("o"); nb = obs.alias("nb")
joined = (o.join(F.broadcast(pairs), "station_id")
           .join(nb, (F.col("nbr_id") == F.col("nb.station_id")) & (F.col("o.time") == F.col("nb.time")))
           .select(F.col("o.station_id").alias("station_id"), F.col("o.time").alias("time"),
                   F.col("o.air_temperature").alias("t"), F.col("nb.air_temperature").alias("t_nbr")))
nbr_diff = joined.groupBy("station_id", "time").agg((F.first("t") - F.percentile_approx("t_nbr", 0.5)).alias("nbr_diff"))
nbr_diff.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (21)
+- ObjectHashAggregate (20)
   +- Exchange (19)
      +- ObjectHashAggregate (18)
         +- Project (17)
            +- SortMergeJoin Inner (16)
               :- Sort (10)
               :  +- Exchange (9)
               :     +- Project (8)
               :        +- BroadcastHashJoin Inner BuildRight (7)
               :           :- Project (3)
               :           :  +- Filter (2)
               :           :     +- Scan parquet  (1)
               :           +- BroadcastExchange (6)
               :              +- Filter (5)
               :                 +- Scan ExistingRDD (4)
               +- Sort (15)
                  +- Exchange (14)
                     +- Project (13)
                        +- Filter (12)
                           +- Scan parquet  (11)


(1) Scan parquet 
Output [4]: [station_id#0, time#1, air_temperature#2, year#18]
Batched: true
Location: InMemoryFileIndex [file:/Users/ashu/Workspace/weather-data

The plan has three **Exchange** steps, and each one is a shuffle: rows written out by key and read back by other tasks. Two move both sides of the station-to-neighbour join so equal station and hour land together (a sort-merge join). The third regroups the joined rows by station and hour for the median. The small pairs table is broadcast to every task instead, so it never shuffles. On one machine all of this happens in a few seconds. On a cluster the shuffles are where the time and the network traffic go, and they're what Spark is built to do well. It plans them, spills to disk when memory runs short, and can keep shuffle files available when a worker dies.

In [5]:
t0 = time.time()
out = nbr_diff.toPandas()
print(f"{len(out):,} rows in {time.time() - t0:.0f} s"); out.nbr_diff.describe()

1,242,696 rows in 6 s


count    1.242696e+06
mean     1.372989e-01
std      3.073059e+00
min     -3.580000e+01
25%     -1.100000e+00
50%      0.000000e+00
75%      1.200000e+00
max      2.710000e+01
Name: nbr_diff, dtype: float64

### Window functions and reusing pandas code

Many QA checks look along one station's own record, like "this hour versus the 24 before it". Spark's **window functions** do that without a shuffle beyond grouping by station. When the logic is easier in pandas, `applyInPandas` hands each group to a Python function as a pandas DataFrame, so the phase 1 code can be reused at the cost of moving data between Java and Python (through Apache Arrow).

In [6]:
w = Window.partitionBy("station_id").orderBy(F.col("time").cast("long")).rangeBetween(-24 * 3600, -1)
stuck_like = obs.withColumn("prev_max", F.max("air_temperature").over(w)) \
                .withColumn("prev_min", F.min("air_temperature").over(w)) \
                .withColumn("flat_24h", F.col("prev_max") == F.col("prev_min"))

def robust_z(pdf: pd.DataFrame) -> pd.DataFrame:
    med = pdf.air_temperature.median(); mad = (pdf.air_temperature - med).abs().median() * 1.4826
    return pdf.assign(z=(pdf.air_temperature - med) / (mad or 1.0))[["station_id", "time", "z"]]

z = obs.groupBy("station_id").applyInPandas(robust_z, schema="station_id string, time timestamp, z double")
print(stuck_like.where("flat_24h").count(), "hours flat for the previous 24 h;", z.where(F.abs("z") > 6).count(), "with |z| > 6")

824 hours flat for the previous 24 h; 212 with |z| > 6


In [7]:
spark.stop()

## Part 2 · Beam

### The standardize step as a Beam pipeline

A Beam pipeline is a graph of **transforms** applied to PCollections. `Create` makes a PCollection from a list, `Map` and `ParDo` apply a function to each element, and the pipeline runs when the `with` block exits. On the local DirectRunner it's slow and meant for testing. The same code would run on Dataflow or Flink by changing one option.

In [8]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from wxlab import pipeline as pl

RAW, OUT = Path(DATA_DIR) / "raw" / "isd", Path(DATA_DIR) / "work" / "beam" / "standardized"
units = pl.units_for(sorted(p.stem for p in (STD / "year=2023").glob("*.parquet"))[:20], [2023])

opts = PipelineOptions(flags=[], runner="DirectRunner", direct_num_workers=4, direct_running_mode="multi_threading")
with beam.Pipeline(options=opts) as p:
    (p | "units" >> beam.Create(units)
       | "standardize" >> beam.Map(pl.run_unit, RAW, OUT)
       | "ok?" >> beam.Map(lambda row: (row["status"], 1))
       | "count" >> beam.CombinePerKey(sum)
       | "print" >> beam.Map(print))

('ok', 20)


The unit function is the same one Ray ran. What Beam adds isn't visible in a batch job this small. It's the model for time.

### Event time, windows and late data

In a live feed each station-hour is an element with an **event time** (when it was measured), and it arrives at some **processing time** (when the pipeline sees it). Beam groups elements into **windows** by event time, for example one window per UTC day for daily checks. It tracks a **watermark**, its estimate of how far event time has progressed, to decide when a window is complete enough to emit. Data that arrives after the watermark has passed its window is **late**. With `allowed_lateness` Beam re-fires the window with the late data included, which is exactly "verdicts change when late data lands" from notebook 06.

`TestStream` lets us script a stream with controlled arrival times on the DirectRunner: two days of readings from one station, then the watermark moves past day one, then a late reading for day one arrives.

In [9]:
from apache_beam.testing.test_stream import TestStream
from apache_beam.transforms import window, trigger
from apache_beam.options.pipeline_options import StandardOptions

d1, d2 = pd.Timestamp("2024-02-05", tz="UTC").timestamp(), pd.Timestamp("2024-02-06", tz="UTC").timestamp()
stream = (TestStream()
          .add_elements([window.TimestampedValue(("KSFO", 12.0), d1 + 3600 * h) for h in range(0, 24, 6)])
          .add_elements([window.TimestampedValue(("KSFO", 14.0), d2 + 3600 * h) for h in range(0, 24, 6)])
          .advance_watermark_to(d2 + 3600)                  # day one is now "complete"
          .add_elements([window.TimestampedValue(("KSFO", 30.0), d1 + 3600 * 23)])   # a late, odd reading for day one
          .advance_watermark_to_infinity())

TIMING = {0: "EARLY", 1: "ON_TIME", 2: "LATE", 3: "UNKNOWN"}
opts = PipelineOptions(flags=[]); opts.view_as(StandardOptions).streaming = True
with beam.Pipeline(options=opts) as p:
    (p | stream
       | beam.WindowInto(window.FixedWindows(24 * 3600), allowed_lateness=2 * 24 * 3600,
                         trigger=trigger.AfterWatermark(late=trigger.AfterCount(1)),
                         accumulation_mode=trigger.AccumulationMode.ACCUMULATING)
       | beam.GroupByKey()
       | beam.Map(lambda kv, w=beam.DoFn.WindowParam, pane=beam.DoFn.PaneInfoParam:
                  print(f"{w.start.to_utc_datetime():%Y-%m-%d}  pane {pane.index} {TIMING[pane.timing]:<8}"
                        f"{len(list(kv[1]))} readings, max {max(kv[1]):.0f} °C")))

2024-02-05  pane 0 ON_TIME 4 readings, max 12 °C
2024-02-05  pane 1 LATE    5 readings, max 30 °C
2024-02-06  pane 0 ON_TIME 4 readings, max 14 °C


Day one fires once ON_TIME, when the watermark passes midnight, with four readings and a maximum of 12 °C. Then the late reading arrives and day one fires again as a LATE pane with five readings and a maximum of 30 °C. Because the window accumulates, the late pane is the complete new answer, not just the difference. Day two fires once when the stream ends.

That's the mechanism behind notebook 06's verdict churn, made explicit. A daily check downstream would see day one twice and must treat the second result as version 2 of the same day. That's only safe if its writes are keyed by station and day and simply overwrite, which is the idempotent-unit rule again.

Spark has event-time watermarks too, in Structured Streaming, but there one watermark does two jobs. It sets how late data may be and when Spark drops the state for a window ([Databricks](https://docs.databricks.com/structured-streaming/watermarks.html)). Beam keeps four separate choices: the watermark, the allowed lateness, the trigger that decides when to emit, and whether later panes accumulate or replace earlier ones ([comparison](https://www.codecentric.de/en/knowledge-hub/blog/event-time-processing-apache-spark-apache-flink)). Flink implements the same model natively, which is why Beam on Flink is a common choice for live pipelines. Ray has none of this built in.

One caution: Python's local runner has an open issue where its panes and late-data handling don't fully match the documented model ([BEAM-7825](https://issues.apache.org/jira/browse/BEAM-7825)). It's fine for learning the ideas. Test a real streaming pipeline on the runner it will run on.

### Beam for grids

Beam also underpins **xarray-beam**, which runs xarray computations as Beam pipelines over chunks of big arrays. It's how WeatherBench 2 computes its scores over terabytes of forecasts. Notebook 11 uses it to rechunk the HRRR cube.

## How the three compare

| | Ray | Spark | Beam |
|---|---|---|---|
| Core idea | Python tasks and actors | Tables and SQL-style queries | Collections over event time, any runner |
| Natural fit | ML pipelines, mixed and stateful work, GPUs | Large joins, aggregations and shuffles, SQL | Streaming with windows and late data; batch on the same code |
| Awkward at | Big shuffles and joins, event-time windows | Arbitrary Python per row (crosses into Python), stateful actors | Interactive work; local runner is slow; debugging through a runner |
| Fault tolerance | Task retries from lineage; actors need care | Stage retries; shuffle files survive worker loss | Delegated to the runner (Dataflow, Flink checkpoints) |
| In this lab | Notebook 07: units, manifest, actors | Neighbour join, windows, pandas UDF | Same unit function; late data re-fires a day |

## What surprised me

**The neighbour check is three shuffles.** One line of pandas in notebook 05 becomes two join shuffles and a group-by shuffle once data is split across machines.

**Spark is a Java program first.** Its memory lives in a Java heap that has to be sized up front, and its workers have to run exactly the same Python as the notebook or they refuse to start.

**Beam's late pane is a complete answer.** With accumulating panes the late firing for day one carried all five readings, not just the new one. That makes "overwrite by key" the right way to handle it downstream.

**The local runners are for learning, not scale.** Every framework here ran the same job on a laptop. Each one's real strengths only show on a cluster.

## Check yourself

<details><summary>Why does the neighbour difference need a shuffle but the 24-hour window doesn't?</summary>
The window only looks within one station, and the data can be partitioned by station once. The neighbour difference needs different stations' readings for the same hour together, which needs a regroup by time.
</details>

<details><summary>What does allowed_lateness change about a daily window?</summary>
It keeps the window's state after the watermark passes, so late readings within the allowance trigger a new result for that day instead of being dropped.
</details>

<details><summary>When would you choose Beam over Spark for this pipeline?</summary>
When the main path is a live feed where data arrives late or out of order and results must be updated by event time, and when you want the same code for the historical backfill.
</details>

New terms from this notebook are in [`docs/glossary.md`](../docs/glossary.md).